In [ ]:
"""
import os
from pathlib import Path

# 1. Get the current directory
current_dir = os.getcwd()

# 2. If we are in 'research', move up one level
if "research" in current_dir:
    os.chdir("..")
    print(f"Moved from research to root: {os.getcwd()}")
else:
    print(f"Already in root: {os.getcwd()}")

# 3. VERIFY: List the files to see if 'config' is visible
if Path("config").exists():
    print("✅ SUCCESS: The 'config' folder is visible from here.")
else:
    print("❌ ERROR: Still can't see 'config'. Check your folder structure.")

"""

Moved from research to root: c:\Users\Aswin\MLprojects\Quality_Of_wine(EndtoEnd ML)\Quality_Of_Wine-EndtoEnd-
✅ SUCCESS: The 'config' folder is visible from here.


In [2]:
""" 
from the config.yaml(where the data ingestion artifact file is resent) 
the path is directed 

this is an ENTITY, a return type for a function

""" 
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [3]:
from Quality_of_Wine.constants import *
from Quality_of_Wine.utils.common import read_yaml,create_directories

In [4]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

In [5]:
import os
import urllib.request as request
import zipfile
from Quality_of_Wine import logger

from Quality_of_Wine.utils.common import get_size
#this is specifically used to know the file size

In [6]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")



    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
  

In [7]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-03-01 22:27:52,970: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-01 22:27:52,981: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-01 22:27:52,983: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-03-01 22:27:52,987: INFO: common: created directory at: artifacts]
[2026-03-01 22:27:52,989: INFO: common: created directory at: artifacts/data_ingestion]
[2026-03-01 22:27:55,457: INFO: 2366278714: artifacts/data_ingestion/data.zip download! with following info: 
Connection: close
Content-Length: 23329
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "c69888a4ae59bc5a893392785a938ccd4937981c06ba8a9d6a21aa52b4ab5b6e"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: A7E4:321A57:178E7F:4150E8:69A4700F
Accept-Ranges: bytes
Date: Sun, 01